[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/langgraph_agent_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪 LangGraph 智能体</h1>

LangGraph 提供方便定义结构化 AI 智能体的工具，但理解内部执行过程和定位问题往往并不容易。Phoenix 将查询引擎每次调用的底层结构可视化，并根据延迟、token 数及其他评估指标展示存在问题的 span，让 LLM 应用具备*可观测性*。

本教程将：
- 使用 LangGraph 构建简单的 SQL 数据库智能体
- 以 OpenInference 格式记录追踪数据
- 检查应用的追踪和 span，定位延迟和成本来源

ℹ️ 本 Notebook 需要 OpenAI API 密钥。

## 准备环境

首先安装所需包并设置 API 密钥。

In [ ]:
%pip install -U langgraph langchain-openai langchain-community arize-phoenix openinference-instrumentation-langchain

In [ ]:
import getpass
import os


def _set_env(key: str):
    if key not in os.environ:
        os.environ[key] = getpass.getpass(f"{key}:")


_set_env("OPENAI_API_KEY")

## 配置数据库

本教程创建 SQLite 数据库。SQLite 轻量且易于配置和使用。这里加载表示数字媒体商店的示例数据库 `chinook`，更多信息见[此处](https://www.sqlitetutorial.net/sqlite-sample-database/)。

为方便使用，我们将数据库 `Chinook.db` 托管在公开 GCS 存储桶中。

In [ ]:
import requests

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"

response = requests.get(url)

if response.status_code == 200:
    # 以二进制写模式打开本地文件
    with open("Chinook.db", "wb") as file:
        # 将响应内容（文件）写入本地文件
        file.write(response.content)
    print("文件已下载并保存为 Chinook.db")
else:
    print(f"文件下载失败，状态码： {response.status_code}")

使用 `langchain_community` 包中的 SQL 数据库封装与数据库交互。它提供执行 SQL 查询和获取结果的简单接口。稍后还会使用 `langchain_openai` 包调用 OpenAI 语言模型 API。

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")
print(db.dialect)
print(db.get_usable_table_names())
db.run("SELECT * FROM Artist LIMIT 10;")

## 工具函数

定义一些支持智能体实现的工具函数。具体来说，为 `ToolNode` 包装回退逻辑，以处理错误并将其反馈给智能体。

In [ ]:
from typing import Any

from langchain_core.messages import ToolMessage
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks
from langgraph.prebuilt import ToolNode


def create_tool_node_with_fallback(tools: list) -> RunnableWithFallbacks[Any, dict]:
    """
    创建带回退机制的 ToolNode，处理错误并将其提供给智能体。
    """
    return ToolNode(tools).with_fallbacks(
        [RunnableLambda(handle_tool_error)], exception_key="error"
    )


def handle_tool_error(state) -> dict:
    error = state.get("error")
    tool_calls = state["messages"][-1].tool_calls
    return {
        "messages": [
            ToolMessage(
                content=f"Error: {repr(error)}\n 请修正错误。",
                tool_call_id=tc["id"],
            )
            for tc in tool_calls
        ]
    }

## 为智能体定义工具

定义以下数据库交互工具：

1. `list_tables_tool`：获取数据库中的可用表
2. `get_schema_tool`：获取表的 DDL
3. `db_query_tool`：执行查询并返回结果，查询失败时返回错误信息

前两个工具从 `langchain_community` 包的 `SQLDatabaseToolkit` 获取。

In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI

toolkit = SQLDatabaseToolkit(db=db, llm=ChatOpenAI(model="gpt-4o"))
tools = toolkit.get_tools()

list_tables_tool = next(tool for tool in tools if tool.name == "sql_db_list_tables")
get_schema_tool = next(tool for tool in tools if tool.name == "sql_db_schema")

print(list_tables_tool.invoke(""))

print(get_schema_tool.invoke("Artist"))

第三个工具手动定义。`db_query_tool` 在数据库中执行查询并返回结果。

In [ ]:
from langchain_core.tools import tool


@tool
def db_query_tool(query: str) -> str:
    """
    对数据库执行 SQL 查询并获取结果。
    若查询不正确，将返回错误消息。
    若返回错误，请重写并检查查询，然后重试。
    """
    result = db.run_no_throw(query)
    if not result:
        return "错误：查询失败。请重写查询并重试。"
    return result


print(db_query_tool.invoke("SELECT * FROM Artist LIMIT 10;"))

虽然严格来说不是工具，但接下来会提示 LLM 检查查询中的常见错误，并将其作为工作流节点。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

query_check_system = """你是注重细节的 SQL 专家。
请再次检查 SQLite 查询中的常见错误，包括：
- 对包含 NULL 的值使用 NOT IN
- 应使用 UNION ALL 时使用 UNION
- 使用 BETWEEN 表示不含边界的范围
- 谓词中的数据类型不匹配
- 标识符的引号是否正确
- 函数参数数量是否正确
- 是否转换为正确数据类型
- 连接是否使用正确列

若有上述错误，请重写查询；否则原样返回查询。

完成检查后，调用适当工具执行查询。"""

query_check_prompt = ChatPromptTemplate.from_messages(
    [("system", query_check_system), ("placeholder", "{messages}")]
)
query_check = query_check_prompt | ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(
    [db_query_tool], tool_choice="required"
)

query_check.invoke({"messages": [("user", "SELECT * FROM Artist LIMIT 10;")]})

## 定义工作流

现在定义智能体工作流。首先强制调用 `list_tables_tool` 获取可用表，然后按本教程开头介绍的步骤继续。

<div class="admonition note">
    <p class="admonition-title">在 LangChain 中使用 Pydantic</p>
    <p>
        本 Notebook 使用 Pydantic v2 的 <code>BaseModel</code>，要求 <code>langchain-core >= 0.3</code>。使用 <code>langchain-core < 0.3</code> 会因混用 Pydantic v1 和 v2 的 <code>BaseModels</code> 而报错。
    </p>
</div>

In [ ]:
from typing import Annotated, Literal

from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import AnyMessage, add_messages
from pydantic import BaseModel, Field
from typing_extensions import TypedDict


# 定义智能体状态
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


# 定义新图
workflow = StateGraph(State)


# 添加首次工具调用的节点
def first_tool_call(state: State) -> dict[str, list[AIMessage]]:
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "sql_db_list_tables",
                        "args": {},
                        "id": "tool_abcd123",
                    }
                ],
            )
        ]
    }


def model_check_query(state: State) -> dict[str, list[AIMessage]]:
    """
    在执行前使用此工具再次检查查询是否正确。
    """
    return {"messages": [query_check.invoke({"messages": [state["messages"][-1]]})]}


workflow.add_node("first_tool_call", first_tool_call)

# 添加前两个工具的节点
workflow.add_node("list_tables_tool", create_tool_node_with_fallback([list_tables_tool]))
workflow.add_node("get_schema_tool", create_tool_node_with_fallback([get_schema_tool]))

# 添加模型节点，根据问题和可用表选择相关表
model_get_schema = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools([get_schema_tool])
workflow.add_node(
    "model_get_schema",
    lambda state: {
        "messages": [model_get_schema.invoke(state["messages"])],
    },
)


# 描述用于表示结束状态的工具
class SubmitFinalAnswer(BaseModel):
    """根据查询结果向用户提交最终答案。"""

    final_answer: str = Field(..., description="给用户的最终答案")


# 添加模型节点，根据问题和模式生成查询
query_gen_system = """你是注重细节的 SQL 专家。

根据输入问题，输出语法正确的 SQLite 查询并执行，随后查看查询结果并返回答案。

提交最终答案时，除了 SubmitFinalAnswer，不得调用其他工具。

生成查询时：

直接输出回答输入问题的 SQL 查询，不进行工具调用。

除非用户指定需要的示例数量，否则查询最多返回 5 条结果。
可以按相关列排序，以返回数据库中最有价值的示例。
不要查询某张表的全部列，只选择与问题相关的列。

若执行查询出错，请重写并重试。

若得到空结果集，请尝试重写查询以获得非空结果。
信息不足时绝不编造，只需说明没有足够信息。

若已有足够信息回答输入问题，直接调用适当工具向用户提交最终答案。

不得对数据库执行任何 DML 语句（INSERT、UPDATE、DELETE、DROP 等）。"""
query_gen_prompt = ChatPromptTemplate.from_messages(
    [("system", query_gen_system), ("placeholder", "{messages}")]
)
query_gen = query_gen_prompt | ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(
    [SubmitFinalAnswer]
)


def query_gen_node(state: State):
    message = query_gen.invoke(state)

    # LLM 有时会产生幻觉并调用错误工具，需要捕获并返回错误消息。
    tool_messages = []
    if message.tool_calls:
        for tc in message.tool_calls:
            if tc["name"] != "SubmitFinalAnswer":
                tool_messages.append(
                    ToolMessage(
                        content=f"错误：调用了错误的工具： {tc['name']}。请修正错误。记住仅调用 SubmitFinalAnswer 提交最终答案；生成的查询必须直接输出，不进行工具调用。",
                        tool_call_id=tc["id"],
                    )
                )
    else:
        tool_messages = []
    return {"messages": [message] + tool_messages}


workflow.add_node("query_gen", query_gen_node)

# 添加模型节点，在执行前检查查询
workflow.add_node("correct_query", model_check_query)

# 添加执行查询的节点
workflow.add_node("execute_query", create_tool_node_with_fallback([db_query_tool]))


# 定义条件边，决定继续还是结束工作流
def should_continue(state: State) -> Literal[END, "correct_query", "query_gen"]:
    messages = state["messages"]
    last_message = messages[-1]
    # 如果存在工具调用，则结束
    if getattr(last_message, "tool_calls", None):
        return END
    if last_message.content.startswith("Error:"):
        return "query_gen"
    else:
        return "correct_query"


# 指定节点之间的边
workflow.add_edge(START, "first_tool_call")
workflow.add_edge("first_tool_call", "list_tables_tool")
workflow.add_edge("list_tables_tool", "model_get_schema")
workflow.add_edge("model_get_schema", "get_schema_tool")
workflow.add_edge("get_schema_tool", "query_gen")
workflow.add_conditional_edges(
    "query_gen",
    should_continue,
)
workflow.add_edge("correct_query", "execute_query")
workflow.add_edge("execute_query", "query_gen")

# 将工作流编译为可运行对象
app = workflow.compile()

## 可视化图

In [ ]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        app.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

### 为智能体自动添加埋点

本示例使用托管 Phoenix 实例。如尚未创建，请在[此处](https://phoenix.arize.com)免费创建并获取 API 密钥。

如需自托管 Phoenix，请按照[此处说明](https://arize.com/docs/phoenix/deployment)操作。

In [ ]:
_set_env("PHOENIX_API_KEY")

In [ ]:
from openinference.instrumentation.langchain import LangChainInstrumentor
from phoenix.otel import register

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"
os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={os.getenv('PHOENIX_API_KEY')}"

tracer_provider = register()
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

## 运行智能体

In [ ]:
messages = app.invoke({"messages": [("user", "Which albums did Aerosmith release?")]})
json_str = messages["messages"][-1].tool_calls[0]["args"]["final_answer"]
json_str

## 后续步骤

现在应能在 [Phoenix 界面](https://app.phoenix.arize.com)中看到追踪！

追踪包含工具调用、LLM 调用和整体执行路径等信息。

接下来可以利用已捕获的追踪：
- 可视化智能体执行路径，追踪 token 用量、延迟和错误
- [评估智能体执行的各项指标](https://arize.com/docs/phoenix/evaluation/how-to-evals/evaluating-phoenix-traces)，例如工具输出质量和整体路径
- [添加标注](https://arize.com/docs/phoenix/tracing/how-to-tracing/capture-feedback#send-annotations-to-phoenix)，记录每次执行的反馈或额外上下文
- 利用标注和评估[创建数据集](https://arize.com/docs/phoenix/datasets-and-experiments/how-to-datasets/creating-datasets)，用于[实验](https://arize.com/docs/phoenix/datasets-and-experiments/how-to-experiments/run-experiments)或导出进行[微调](https://arize.com/docs/phoenix/datasets-and-experiments/how-to-datasets/exporting-datasets)